# Домашняя работа 3

## Подготовка

In [1]:
from pathlib import Path
import pandas as pd
from pandas.testing import assert_frame_equal
DATA = Path("../data")
WORK = Path("../results")
WORK.mkdir(exist_ok=True)
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
print("pandas", pd.__version__, decisions.shape)

pandas 2.2.3 (6830, 7)


## Исходники домашней работы

In [2]:
org_keys = ["lei", "activity_year"]
long_keys = org_keys + ["action_taken"]
organizations = pd.read_csv(DATA / "organizations.csv", dtype={"lei": "string"})
lookup = organizations[org_keys + ["organization_name"]].copy()
assert not lookup.duplicated(org_keys).any()
print(organizations.head(3))

                    lei                       organization_name  source_count  \
0  549300AG64NHILB7ZP05                      LOANDEPOT.COM, LLC           114   
1  QFROUN1UWUYU0DVIWD51  Fifth Third Bank, National Association             3   
2  5493003GQDUH26DNNH17               NAVY FEDERAL CREDIT UNION           742   

   activity_year  
0           2023  
1           2023  
2           2023  


## H1 Профиль отказов

In [3]:
decisions_h1 = decisions.copy()
decisions_h1["is_denied"] = (decisions_h1["action_taken"] == 3).astype(int)

denial = decisions_h1.groupby(org_keys).agg(
    n_decisions=("action_taken", "size"),
    n_denied=("is_denied", "sum")
).reset_index()

denial["denial_rate"] = denial["n_denied"] / denial["n_decisions"]

home_denial_profile = denial.merge(
    lookup, on=org_keys, how="left", validate="one_to_one", indicator=True
)

assert home_denial_profile["_merge"].eq("both").all()
assert home_denial_profile["n_decisions"].sum() == len(decisions)
assert home_denial_profile["n_denied"].sum() == (decisions["action_taken"] == 3).sum()
assert home_denial_profile["denial_rate"].between(0, 1).all()

home_denial_profile = home_denial_profile.drop(columns=["_merge"])
home_denial_profile.to_csv(WORK / "home_denial_profile.csv", index=False)

## H2 Группы для просмотра

In [4]:
large_groups = home_denial_profile[home_denial_profile["n_decisions"].ge(20)].copy()
large_groups.to_csv(WORK / "home_denial_groups20.csv", index=False)

print(len(home_denial_profile), len(large_groups))

overall_denial_rate = home_denial_profile["n_denied"].sum() / home_denial_profile["n_decisions"].sum()
print(overall_denial_rate)

314 61
0.10117130307467057


## H3 Длинная, широкая и снова длинная форма

In [6]:
long_h3 = decisions.groupby(long_keys, dropna=False).size().rename("n").reset_index()
wide_h3 = long_h3.pivot(index=org_keys, columns="action_taken", values="n")

value_vars_h3 = [1, 2, 3]
wide_h3 = wide_h3.reindex(columns=value_vars_h3).fillna(0).astype("int64").reset_index()

melted_h3 = wide_h3.melt(id_vars=org_keys, value_vars=value_vars_h3, var_name="action_taken", value_name="n")
melted_h3["action_taken"] = melted_h3["action_taken"].astype("int64")
melted_h3 = melted_h3[melted_h3["n"] > 0]

long_sorted = long_h3.sort_values(long_keys).reset_index(drop=True)
melted_sorted = melted_h3.sort_values(long_keys).reset_index(drop=True)

assert_frame_equal(melted_sorted, long_sorted, check_dtype=False)

## H+ Безопасное соединение